# Computer Vision

In [1]:
import torch
import torch.optim as optim
import torch.nn as nn 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


Load dataset, called Fashion MNIST. <br>
Create two sets of data, one for training and one for testing. <br>


In [ ]:
# Load the dataset 
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(root='./data', train=True,
                                      download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False,
                                     download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64,
                          shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64,
                         shuffle=False)

**Init class** has a define method call flatten (nn.Flatten()), <br>
which is a built-in function to flatten the 2d image to 1d. <br>
And another method called linear_relu_stack, a sequence of layer <br>
and operations (aberviated to *ops*) that define the hehaviour of <br>
of the network. <br>
**Forward** we'll simply define how these work. <br>
First, X is flatten data, by calling self.flatten. <br>
Then the results is passed to linear_relu_stack, <br>
called logits which are log probabilities (defined by LogSoftmax). 

In [ ]:
#Define the model 
class FashionMNISTModel(nn.Module):
    
    def __init__(self):
        super(FashionMNISTModel,self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128,10),
            nn.LogSoftmax(dim=1)
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
model = FashionMNISTModel()

First the **loss function**, which is defined as nn>NLLLoss(), <br>
which stand for *Negative Log Likelihood Loss*. Works well with <br>
log probabilities output with *logits*. <br>
Secondly is the **Optimizer**, which uses the Adam optimization <br>
algorithm. And the parameter as model.parameters(), allowing all the <br>
trainable parameters in the model to the optimizer, to help minimize the <br>
loss calculated by the loss function. 

In [ ]:
#Define the loss function and optimizer
loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters())



In [ ]:
#Train the model
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x,y) in enumerate(dataloader):
        #compute prediction and loss 
        pred = model(x)
        loss = loss_fn(pred,y)
        
        #Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [ ]:
#Training process 
epochs = 5 
for t in range(epochs):
    print(f"Epoch {t+1}\n-----------------------")
    train(train_loader, model, loss_function, optimizer)
print("Done")